In [7]:
import pandas as pd

In [5]:
# Getting URL for green_tripdata 2025
url =  'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [8]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID', 
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount']

df = pd.read_parquet(url, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [17]:
# Creating a Topoc
!docker exec -it redpanda rpk topic create green-trips

TOPIC        STATUS
green-trips  OK


In [18]:
ride = df.iloc[0]
ride



lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [19]:
from model import Ride, ride_from_row, ride_serializer, dataclasses

In [20]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [21]:
topic_name = "green-trips"

ride_obj = ride_from_row(ride)

producer.send(topic_name, value=ride_obj)
producer.flush()


In [ ]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

In [15]:
# checking the list of topics
! docker exec -it redpanda rpk topic list 

NAME         PARTITIONS  REPLICAS
green-trips  1           1
rides        1           1


In [16]:
# deleting the topic
! docker exec -it redpanda rpk topic delete green-trips

TOPIC        STATUS
green-trips  OK
